# Agent 1 – Ingestion & Pipeline RAG

Cet agent extrait le texte des PDFs, les découpe en chunks, génère des embeddings via Gemini et les indexe dans ChromaDB.

In [1]:
# ============================================================
# Imports
# ============================================================
import pdfplumber
import chromadb
from llm_adapter import GeminiEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import json
import time
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path(".env"))

print("Imports réussis.")
print(f"ChromaDB version : {chromadb.__version__}")

Imports réussis.
ChromaDB version : 1.5.9


In [2]:
# ============================================================
# Configuration
# ============================================================

PAPERS_DIR    = "papers"         # dossier des PDFs (Source/papers/)
CHROMA_DB_PATH = "../chroma_db"  # base vectorielle (racine du projet)
LOGS_DIR      = "../logs"        # logs (racine du projet)
COLLECTION_NAME = "literature_review"
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 150
BATCH_SIZE    = 50

print(f"PAPERS_DIR      : {Path(PAPERS_DIR).resolve()}")
print(f"CHROMA_DB_PATH  : {Path(CHROMA_DB_PATH).resolve()}")
print(f"COLLECTION_NAME : {COLLECTION_NAME}")

PAPERS_DIR      : C:\Users\hp\projet_revue_litterature\Source\papers
CHROMA_DB_PATH  : C:\Users\hp\projet_revue_litterature\chroma_db
COLLECTION_NAME : literature_review


In [3]:
# ============================================================
# Initialisation ChromaDB
# ============================================================

embedding_fn = GeminiEmbeddingFunction()

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"description": "Chunks d'articles scientifiques"}
)

print(f"Collection '{COLLECTION_NAME}' prête.")
print(f"Documents actuels : {collection.count()}")

Collection 'literature_review' prête.
Documents actuels : 0


In [4]:
# ============================================================
# Extraction du texte PDF
# ============================================================

def extract_text_from_pdf(pdf_path: Path) -> dict:
    result = {
        "full_text": "",
        "metadata": {
            "filename": pdf_path.name,
            "total_pages": 0,
            "file_size_kb": round(pdf_path.stat().st_size / 1024, 2)
        },
        "success": False,
        "error": None
    }
    try:
        pages_text = []
        with pdfplumber.open(pdf_path) as pdf:
            result["metadata"]["total_pages"] = len(pdf.pages)
            for i, page in enumerate(pdf.pages):
                text = page.extract_text()
                if text:
                    pages_text.append(f"[Page {i+1}]\n{text.replace(chr(0), '').strip()}")
        result["full_text"] = "\n\n".join(pages_text)
        result["success"] = True
    except Exception as e:
        result["error"] = str(e)
        print(f"  [ERREUR] {pdf_path.name} : {e}")
    return result

print("Fonction extract_text_from_pdf définie.")

Fonction extract_text_from_pdf définie.


In [5]:
# ============================================================
# Découpage en chunks
# ============================================================

def chunk_text(text: str, metadata: dict) -> list:
    if not text or not text.strip():
        return []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    raw_chunks = splitter.split_text(text)
    chunks = []
    for idx, chunk in enumerate(raw_chunks):
        if len(chunk.strip()) >= 50:
            chunks.append({
                "text": chunk.strip(),
                "metadata": {**metadata, "chunk_index": idx, "total_chunks": len(raw_chunks)}
            })
    return chunks

print("Fonction chunk_text définie.")

Fonction chunk_text définie.


In [6]:
# ============================================================
# Ingestion d'un PDF dans ChromaDB
# ============================================================

def ingest_paper(pdf_path: Path) -> dict:
    paper_id = pdf_path.stem.lower().replace(" ", "_").replace("-", "_")
    result = {
        "paper_id": paper_id,
        "filename": pdf_path.name,
        "chunks_created": 0,
        "total_pages": 0,
        "status": "pending",
        "error": None
    }
    try:
        print(f"  Traitement : {pdf_path.name}")

        extraction = extract_text_from_pdf(pdf_path)
        if not extraction["success"] or not extraction["full_text"].strip():
            result["status"] = "error"
            result["error"] = extraction.get("error", "Texte vide")
            return result

        result["total_pages"] = extraction["metadata"]["total_pages"]
        chunks = chunk_text(extraction["full_text"], extraction["metadata"])

        if not chunks:
            result["status"] = "skipped"
            result["error"] = "Aucun chunk valide"
            return result

        print(f"    {len(chunks)} chunks depuis {result['total_pages']} pages")

        for batch_start in range(0, len(chunks), BATCH_SIZE):
            batch = chunks[batch_start: batch_start + BATCH_SIZE]
            collection.upsert(
                ids=[f"{paper_id}_chunk_{c['metadata']['chunk_index']}" for c in batch],
                documents=[c["text"] for c in batch],
                metadatas=[{k: str(v) if not isinstance(v, (str, int, float, bool)) else v
                            for k, v in c["metadata"].items()} for c in batch]
            )

        result["chunks_created"] = len(chunks)
        result["status"] = "success"
        print(f"    OK : {len(chunks)} chunks indexés")

    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)
        print(f"    [ERREUR] {e}")

    return result

print("Fonction ingest_paper définie.")

Fonction ingest_paper définie.


In [7]:
# ============================================================
# Exécution de l'Agent d'Ingestion
# ============================================================

start_time = time.time()
print("=" * 60)
print("AGENT 1 – INGESTION & PIPELINE RAG")
print("=" * 60)

papers_path = Path(PAPERS_DIR).resolve()
pdf_files = list(papers_path.glob("*.pdf"))

print(f"Dossier PDFs : {papers_path}")
print(f"PDFs trouvés : {len(pdf_files)}")
print()

if not pdf_files:
    print(f"[ATTENTION] Aucun PDF trouvé dans {papers_path}")
else:
    results = []
    for pdf_path in pdf_files:
        results.append(ingest_paper(pdf_path))

    # Statistiques
    successful = [r for r in results if r["status"] == "success"]
    total_chunks = sum(r["chunks_created"] for r in results)
    duration = round(time.time() - start_time, 2)

    # Log JSON
    logs_dir = Path(LOGS_DIR)
    logs_dir.mkdir(parents=True, exist_ok=True)
    summary = {
        "status": "completed",
        "total_papers": len(pdf_files),
        "successful": len(successful),
        "failed": len(results) - len(successful),
        "total_chunks": total_chunks,
        "duration_seconds": duration,
        "completed_at": datetime.now().isoformat(),
        "papers": results
    }
    (logs_dir / "ingestion_log.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    # Tableau récapitulatif
    print()
    print("=" * 60)
    print("RÉSUMÉ")
    print("=" * 60)
    print(f"{'Article':<38} {'Statut':<8} {'Chunks'}")
    print("-" * 58)
    for r in results:
        status = "OK" if r["status"] == "success" else "ERR"
        name = r["filename"][:36] + ".." if len(r["filename"]) > 38 else r["filename"]
        print(f"{name:<38} {status:<8} {r['chunks_created']}")
    print("-" * 58)
    print(f"TOTAL : {len(successful)}/{len(results)} articles | {total_chunks} chunks | {duration}s")
    print(f"ChromaDB : {collection.count()} documents au total")

AGENT 1 – INGESTION & PIPELINE RAG
Dossier PDFs : C:\Users\hp\projet_revue_litterature\Source\papers
PDFs trouvés : 5

  Traitement : attention_is_all_you_need.pdf


    58 chunks depuis 15 pages


    [ERREUR] 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}} in upsert.
  Traitement : bert_pre_training.pdf


  Traitement : chain_of_thought_wei2022.pdf


    193 chunks depuis 43 pages


    [ERREUR] 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}} in upsert.
  Traitement : llm_survey_2023.pdf


... (4 outputs left out)

    1303 chunks depuis 144 pages


    [ERREUR] 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}} in upsert.
  Traitement : rag_paper_lewis2020.pdf


    101 chunks depuis 19 pages


    [ERREUR] 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}} in upsert.

RÉSUMÉ
Article                                Statut   Chunks
----------------------------------------------------------
attention_is_all_you_need.pdf          ERR      0
bert_pre_training.pdf                  ERR      0
chain_of_thought_wei2022.pdf           ERR      0
llm_survey_2023.pdf                    ERR      0
rag_paper_lewis2020.pdf                ERR      0
----------------------------------------------------------
TOTAL : 0/5 articles | 0 chunks | 265.02s
ChromaDB : 0 documents au total


In [ ]:
# ============================================================
# Vérification : requête de test
# ============================================================

total_docs = collection.count()
print(f"Documents indexés : {total_docs}")

if total_docs > 0:
    test_query = "attention mechanism transformer"
    print(f"Requête test : '{test_query}'")
    print("-" * 50)
    try:
        res = collection.query(
            query_texts=[test_query],
            n_results=min(3, total_docs),
            include=["documents", "metadatas", "distances"]
        )
        for i, (doc, meta, dist) in enumerate(zip(
            res["documents"][0], res["metadatas"][0], res["distances"][0]
        )):
            print(f"#{i+1} | {meta.get('filename','?')} | distance={dist:.4f}")
            print(f"   {doc[:200]}...")
        print("\nBase vectorielle operationnelle. L'Agent 2 peut demarrer.")
    except Exception as e:
        print(f"[INFO] Requete de test echouee (rate limit probable) : {str(e)[:100]}")
        print(f"L'ingestion a reussi ({total_docs} docs). L'Agent 2 peut demarrer.")
else:
    print("[ATTENTION] Collection vide.")